In [9]:
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import relativedelta

# Read demographics 1
columns_to_keep = ['child_id', 'child_race', 'child_ethnicity', 'mother_edu', 'father_edu', 'family_annualincome']
demographics = pd.read_csv('/home/engaclew/neurogen/data/L3_HIPAA_LENA_cleaned/metadata/Demographics_Interim_26JAN23.csv', sep=',')
demographics = demographics.sort_values(['child_id', 'asmt_date'])
demographics = demographics.groupby('child_id').last().reset_index() # Keep only rows with latest assessment date for most up to date info
demographics['family_annualincome'] = (demographics['family_annualincome']
                                     .str.replace('$', '', regex=False)
                                     .str.replace(',', '', regex=False)
                                     .str.strip())
demographics['family_annualincome'] = pd.to_numeric(demographics['family_annualincome'], errors='coerce')
demographics = demographics[columns_to_keep]

# Read demographics 2
demographics2 =  pd.read_csv('/home/engaclew/neurogen/data/L3_HIPAA_LENA_cleaned/metadata/Demographics_Publication_08JUL21_addedDissIDs.csv', sep=',')
demographics2 = demographics2.sort_values(['child_id', 'asmt_id'])
demographics2 = demographics2.groupby('child_id').last().reset_index()
demographics2.rename({'c_family_annualincome': 'family_annualincome'}, axis=1, inplace=True)
demographics2 = demographics2[['child_id', 'child_race', 'child_ethnicity', 'mother_edu', 'family_annualincome']]
demographics2['father_edu'] = None
demographics = pd.concat([demographics, demographics2], ignore_index=True)
demographics = demographics.drop_duplicates(subset=['child_id'], keep='first')

# Read demographics 3
demographics3 = pd.read_csv('/home/engaclew/neurogen/data/L3_HIPAA_LENA_cleaned/metadata/2024-10-14T18_08_53.625Z_k23_remote_export.csv',sep=',')
demographics3 = demographics3[['customId', 'race', 'ethnicity']]
demographics3['customId'] = pd.to_numeric(demographics3['customId'], errors='coerce')
demographics3 = demographics3.dropna(subset=['customId'])
demographics3['customId'] = demographics3['customId'].astype(int)
demographics3['mother_edu'] = None
demographics3['father_edu'] = None
demographics3['family_income'] = None

demographics3.rename({'customId': 'child_id', 'race': 'child_race', 'ethnicity': 'child_ethnicity'}, axis=1, inplace=True)
print(demographics3['child_id'].unique())

demographics = pd.concat([demographics, demographics3], ignore_index=True)
demographics = demographics.drop_duplicates(subset=['child_id'], keep='first')

def categorize_education(edu_level):
    if pd.isna(edu_level) or edu_level == 'None':
        return 'Missing/None'
    elif edu_level in ['Professional degree', 'Doctorate', 'Doctoral degree']:
        return 'Advanced degree'
    elif edu_level in ['4 year degree', '4-year degree']:
        return 'Bachelor\'s degree'
    elif edu_level in ['2 year degree', 'Some college']:
        return 'Some college'
    elif edu_level in ['High school graduate']:
        return 'High school or less'
    else:
        return edu_level

demographics['mother_edu'] = demographics['mother_edu'].apply(categorize_education)
demographics['father_edu'] = demographics['father_edu'].apply(categorize_education)

def categorize_income_brackets(income):
    if pd.isna(income):
        return 'Missing'
    elif income < 70000:
        return r'Under $70k'
    elif income < 120000:
        return r'$70k-$120k'
    elif income < 200000:
        return r'$120k-$200k'
    else:
        return r'$200k+'

demographics['family_annualincome'] = demographics['family_annualincome'].apply(categorize_income_brackets)

# Extract the list of 50 children who have been annotated along with their diagnostic group
annotations = pd.read_csv('/home/engaclew/neurogen/data/L3_HIPAA_LENA_cleaned/metadata/annotations.csv', sep=',')
annotated_recordings = np.unique(annotations[annotations.set == 'eaf/an1']['recording_filename'].values)
recordings = pd.read_csv('/home/engaclew/neurogen/data/L3_HIPAA_LENA_cleaned/metadata/recordings.csv', sep=',')[['recording_filename', 'date_iso', 'child_id']]
#recordings = recordings[recordings['recording_filename'].isin(annotated_recordings)]
children = pd.read_csv('/home/engaclew/neurogen/data/L3_HIPAA_LENA_cleaned/metadata/children.csv', sep=',')[['child_id', 'child_sex', 'group_id', 'child_dob']]
recordings = recordings.merge(children, on='child_id')
def diff_month(row):
    d1 = datetime.strptime(row['date_iso'], '%Y-%m-%d')
    d2 = datetime.strptime(row['child_dob'], '%Y-%m-%d')
    return (d1.year - d2.year) * 12 + d1.month - d2.month
recordings['age'] = recordings.apply(lambda row: diff_month(row), axis=1)
recordings.drop(['date_iso', 'child_dob'], axis=1, inplace=True)

# Merge our 50 children with their demographics info
recordings['child_id'] = recordings['child_id'].astype(object)
demographics['child_id'] = demographics['child_id'].astype(object)
print("Annotated children not in demographics:", set(recordings['child_id']) - set(demographics['child_id']))
recordings = recordings.merge(demographics, on='child_id', how='left')


[6831 6741 6841 6681 5241 5331 4001 5441 5131 3941 5351 5291 5191 5461
 4071 4051 5041 3931 5071 5361 4011 5231 5201 5181 5371 3901 3921 4031
 4021 4081 5211 3891 3951 5061 9999 5781 3961 6211 6161 6351 6361 6461
 5851 6581 6382 6701 6721 6641 6451 6761 6951 6941 6851 6671 6881 6821
 6921 6212 3871 6511 3991 6791 6781 6971 6891 6651 6811 6541 6901 7011
 7121 7111 6981 6961 6991 5901 7141 7261 5961 6691 6491 6801 6911 6731
 7301 7321 5081 3362 6261 6371 6611 6471 6591 6661 7031 3971 5381 3911
 6391 6561 6771 6431 6411 6601 7101 6481 7051 7041 7061 7091 7151 7171
 7131 7221 7201 7241 7251 7191 7211 7231 7271 7281 7311 7291]
Annotated children not in demographics: {'3891', '5901', '5291', '5041', '3501', '6901', '6651', '6581', '7301', '6321', '6311', '5531', '6421', '6921', '2851', '6361', '6221', '2941', '5261', '7101', '7051', '3641', '3011', '5441', '3921', '7191', '3211', '5131', '7091', '2771', '6991', '2811', '5511', '5221', '6121', '6001', '3621', '4081', '6371', '5351', '2931', '

In [11]:
recordings = recordings[recordings['group_id'].isin(['angelman_syndrome', 'low_risk', 'down_syndrome'

array(['autism_sibling', 'angelman_syndrome', 'low_risk', 'down_syndrome',
       'fragile_x_syndrome', 'autism_spectrum_disorder',
       'environmental_risk'], dtype=object)

In [3]:
def summarize_demographics_by_group(df):
    group_order = ['low_risk', 'angelman_syndrome', 'fragile_x_syndrome', 
                   'down_syndrome', 'autism_sibling']
    
    # Set categorical order on group_id (not group)
    df['group_id'] = pd.Categorical(df['group_id'], categories=group_order, ordered=True)
    
    summary = df.groupby('group_id').agg({
        'age': ['count', 'mean', 'std'],
        'child_sex':  lambda x: x.value_counts(dropna=False).to_dict(),
        'child_race': lambda x: x.value_counts(dropna=False).to_dict(),
        'child_ethnicity': lambda x: x.value_counts(dropna=False).to_dict(),
        'mother_edu': lambda x: x.value_counts(dropna=False).to_dict(),
        'father_edu': lambda x: x.value_counts(dropna=False).to_dict(),
        'family_annualincome': lambda x: x.value_counts(dropna=False).to_dict()
    }).round(2)
    
    return summary

summary = summarize_demographics_by_group(recordings)
summary

/tmp/ipykernel_12942/259134587.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = df.groupby('group_id').agg({


age                     child_sex  \
                   count  mean   std          <lambda>   
group_id                                                 
low_risk              10  20.5  4.84  {'m': 6, 'f': 4}   
angelman_syndrome     10  21.8  3.97  {'f': 6, 'm': 4}   
fragile_x_syndrome    10  21.7  3.71  {'m': 7, 'f': 3}   
down_syndrome         10  22.0  4.45  {'m': 6, 'f': 4}   
autism_sibling        10  20.8  4.89  {'f': 7, 'm': 3}   

                                                           child_race  \
                                                             <lambda>   
group_id                                                                
low_risk                         {'White': 7, None: 2, 'Multiple': 1}   
angelman_syndrome                   {'White': 7, None: 2, 'Asian': 1}   
fragile_x_syndrome                              {'White': 9, None: 1}   
down_syndrome                                {'White': 9, 'Asian': 1}   
autism_sibling      {'White': 8, 'Black / African America': 1, nan...   

                                                      child_ethnicity  \
                                                             <lambda>   
group_id                                                                
low_risk            {'Not Hispanic or Latino': 6, None: 2, 'Non-Hi...   
angelman_syndrome   {'Not Hispanic or Latino': 7, None: 2, 'Hispan...   
fragile_x_syndrome             {'Not Hispanic or Latino': 9, None: 1}   
down_syndrome       {'Not Hispanic or Latino': 9, 'Hispanic or Lat...   
autism_sibling      {'Not Hispanic or Latino': 8, 'Hispanic or Lat...   

                                                           mother_edu  \
                                                             <lambda>   
group_id                                                                
low_risk            {'Advanced degree': 7, 'Bachelor's degree': 2,...   
angelman_syndrome   {'Advanced degree': 4, 'Some college': 2, 'Mis...   
fragile_x_syndrome  {'Advanced degree': 5, 'Some college': 2, 'Bac...   
down_syndrome       {'Advanced degree': 6, 'Bachelor's degree': 3,...   
autism_sibling      {'Some college': 5, 'Bachelor's degree': 2, 'M...   

                                                           father_edu  \
                                                             <lambda>   
group_id                                                                
low_risk            {'Bachelor's degree': 4, 'Advanced degree': 2,...   
angelman_syndrome   {'Advanced degree': 3, 'Bachelor's degree': 3,...   
fragile_x_syndrome  {'Bachelor's degree': 4, 'Some college': 2, 'H...   
down_syndrome       {'Advanced degree': 4, 'Bachelor's degree': 3,...   
autism_sibling      {'Some college': 5, 'Bachelor's degree': 2, 'M...   

                                                  family_annualincome  
                                                             <lambda>  
group_id                                                               
low_risk            {'Under $70k': 5, '$70k-$120k': 4, '$120k-$200...  
angelman_syndrome   {'$70k-$120k': 3, 'Missing': 3, '$200k+': 3, '...  
fragile_x_syndrome  {'$120k-$200k': 4, '$200k+': 3, 'Under $70k': ...  
down_syndrome       {'$70k-$120k': 3, 'Missing': 3, '$120k-$200k':...  
autism_sibling      {'$70k-$120k': 3, 'Under $70k': 3, '$120k-$200...

In [32]:
col = 'family_aincome'
groups = ['low_risk', 'angelman_syndrome', 'fragile_x_syndrome', 'down_syndrome', 'autism_sibling']
for group in groups:
    print(group, summary.loc[group][col].values)

KeyError: 'family_income'

In [108]:
missing_race_children = recordings[recordings['child_race'].isna()]['child_id']
print("Children with missing race data:")
print(missing_race_children.tolist())
print(f"Total children with missing race: {len(missing_race_children)}\n")

missing_income_children = recordings[recordings['family_annualincome'].isna()]['child_id']
print("Children with missing family income data:")
print(missing_income_children.tolist())
print(f"Total children with missing income: {len(missing_income_children)}\n")

missing_income_children = recordings[recordings['mother_edu_years'].isna()]['child_id']
print("Children with mother_edu_years:")
print(missing_income_children.tolist())
print(f"Total children with missing income: {len(missing_income_children)}\n")

Children with missing race data:
['5901', '3461', '6981', '2781', '2761', '7281', '7291', '7141', '5011']
Total children with missing race: 9

Children with missing family income data:
['4011', '5901', '6981', '3941', '5181', '2761', '7281', '7291', '7141', '3131', '5011']
Total children with missing income: 11

Children with mother_edu_years:
['5901', '6261', '3321', '3501', '3461', '3362', '6981', '5061', '2781', '2761', '7281', '7291', '7141', '3871', '5011']
Total children with missing income: 15



In [18]:
import pandas as pd
children = pd.read_csv('/home/engaclew/neurogen/data/L3_HIPAA_LENA_cleaned/metadata/children.csv')[['child_id', 'child_dob', 'child_sex', 'group_id']]
recordings = pd.read_csv('/home/engaclew/neurogen/data/L3_HIPAA_LENA_cleaned/metadata/recordings.csv')
recordings = recordings.merge(children, on='child_id')

def diff_month(row):
    d1 = datetime.strptime(row['date_iso'], '%Y-%m-%d')
    d2 = datetime.strptime(row['child_dob'], '%Y-%m-%d')
    return (d1.year - d2.year) * 12 + d1.month - d2.month
recordings['age'] = recordings.apply(lambda row: diff_month(row), axis=1)
unique_children = recordings.drop_duplicates(subset='child_id')
sex_counts = unique_children['child_sex'].value_counts()
print(f"Males: {sex_counts.get('m', 0)} = {sex_counts.get('m', 0) / len(children)}")
print(f"Females: {sex_counts.get('f', 0)} = {sex_counts.get('f', 0) / len(children)} ")

Males: 117 = 0.5131578947368421
Females: 111 = 0.4868421052631579 


In [15]:
recordings.groupby('chil

,child_id,experiment,date_iso,start_time,recording_device_type,recording_filename,duration,its_filename,child_dob,child_sex,group_id,age
0,7201,neurogen,2022-10-04,10:55:17,lena,20221018_135301_043300_3.wav,766625,20221018_135301_043300_3.its,2020-08-07,f,autism_sibling,26
1,7201,neurogen,2022-10-11,10:19:01,lena,20221018_135301_043300_4.wav,40018062,20221018_135301_043300_4.its,2020-08-07,f,autism_sibling,26
2,3071,neurogen,2018-12-03,9:09:28,lena,20181207_135636_022875_1.wav,1887312,20181207_135636_022875_1.its,2016-08-13,f,angelman_syndrome,28
3,3071,neurogen,2018-12-04,10:05:42,lena,20181207_135636_022875_3.wav,10363187,20181207_135636_022875_3.its,2016-08-13,f,angelman_syndrome,28
4,3071,neurogen,2018-12-04,2:55:50,lena,20181207_135636_022875_4.wav,14995187,20181207_135636_022875_4.its,2016-08-13,f,angelman_syndrome,28
...,...,...,...,...,...,...,...,...,...,...,...,...
376,PO1015,neurogen,2021-12-11,6:44:48,lena,20220118_085259_045737_1.wav,57600000,20220118_085259_045737_1.its,2019-11-08,m,autism_spectrum_disorder,25
377,5031,neurogen,2019-08-14,8:50:07,lena,20190819_210631_022870.wav,57600000,20190819_210631_022870.its,2018-08-19,f,low_risk,12
378,2931,neurogen,2018-01-24,8:31:24,lena,20180206_110905_009463.wav,57600125,20180206_110905_009463.its,2016-12-13,f,low_risk,13
379,3171,neurogen,2018-04-01,8:48:22,lena,20180424_113552_022873.wav,57600125,20180424_113552_022873.its,2017-09-14,f,low_risk,7
